# Inverted Index

The text files are placed in folder './iitfidf', i.e. we have three files in the folder: 1.txt, 2.txt, 3.txt

The **wholeTextFiles** will read the files, and output a tuple (filename, fulltext)

wholeTextFiles -> (filename, fulltext)

In the first step (flatMap), we map our existing key/value pair (filename, fulltext) to a new key-value pair comprising of document-id (i,e, filename) and token (each word in the fulltext) as the key and 1 (representing count) as the value. We will be using the flatMap transformation here to combine all the tokens in a single list.

 flatMap -> ((token, filename), 1)

In the second step (reduceByKey), we will be grouping the key/value pairs with the common key and further aggregate the values for the same key to get the term frequency for a particular word corresponding to its filename.

reduceByKey -> ((token, filename), 1+1+1...)

In the third step, we will change the key/value pairs to a new set of key/ value/pairs with tokens as the key and its filename and respective term frequency as the values.

map -> ((token,count),  filename), notice the count here is also called "Term Frequency"

In [5]:
from pyspark import SparkConf, SparkContext
import os, re

sc = SparkContext.getOrCreate(SparkConf())

#filename has no extension
inverted_index = sc.wholeTextFiles('data/iitfidf')\
                .flatMap(lambda x: [((i.lower(),os.path.basename(x[0]).split(".")[0]) ,1) for i in re.split('\\W', x[1])])\
                .reduceByKey(lambda a, b: a + b)\
                .map(lambda x: ((x[0][0],x[1]),x[0][1]))

#keep file extension           
#inverted_index = sc.wholeTextFiles('data/iitfidf')\
#                .flatMap(lambda x: [((i.lower(),os.path.basename(x[0])), 1) for i in re.split('\\W', x[1])])\
#                .reduceByKey(lambda a, b: a + b)\
#                .map(lambda x: ((x[0][0],x[1]),x[0][1]))

output = inverted_index.take(10)
for i in range(10):
    print(output[i])

sc.stop()

(('knitting', 1), '1')
(('is', 1), '1')
(('my', 3), '1')
(('hobby', 1), '1')
(('and', 2), '1')
(('passion', 1), '1')
(('i', 1), '1')
(('love', 2), '1')
(('dogs', 1), '1')
(('cats', 1), '1')


In [6]:
from pyspark import SparkConf, SparkContext
import os, re

sc = SparkContext.getOrCreate(SparkConf())
#sc = SparkContext("local", "app")
rdd1=sc.wholeTextFiles('data/iitfidf')

rdd2 = rdd1.flatMap(lambda x: [((i.lower(), os.path.basename(x[0]).split(".")[0]) ,1) for i in re.split('\\W', x[1])])\
            .filter(lambda x: x[0][0] != '')
#rdd2 = rdd1.flatMap(lambda x: [((i.lower(),os.path.basename(x[0])), 1) for i in re.split('\\W', x[1])])
    
output = rdd2.take(10)
for i in range(10):
    print(output[i])



(('knitting', '1'), 1)
(('is', '1'), 1)
(('my', '1'), 1)
(('hobby', '1'), 1)
(('and', '1'), 1)
(('my', '1'), 1)
(('passion', '1'), 1)
(('i', '1'), 1)
(('love', '1'), 1)
(('dogs', '1'), 1)


In [7]:
rdd3 = rdd2.reduceByKey(lambda a, b: 1)
output = rdd3.take(10)
for i in range(10):
    print(output[i])


(('knitting', '1'), 1)
(('is', '1'), 1)
(('my', '1'), 1)
(('hobby', '1'), 1)
(('and', '1'), 1)
(('passion', '1'), 1)
(('i', '1'), 1)
(('love', '1'), 1)
(('dogs', '1'), 1)
(('cats', '1'), 1)


In [8]:
inverted_index = rdd3.map(lambda x: (x[0][0],(x[0][1],)))

output = inverted_index.take(10)
for i in range(10):
    print(output[i])
    


('knitting', ('1',))
('is', ('1',))
('my', ('1',))
('hobby', ('1',))
('and', ('1',))
('passion', ('1',))
('i', ('1',))
('love', ('1',))
('dogs', ('1',))
('cats', ('1',))


In [9]:
counts = inverted_index.reduceByKey(lambda a, b: a + b).sortByKey().filter(lambda x: x[0] !='')
result = counts.take(20)
result

[('a', ('2', '3')),
 ('all', ('2',)),
 ('alliance', ('2',)),
 ('and', ('1', '2', '3')),
 ('antagonisms', ('3',)),
 ('are', ('1',)),
 ('as', ('3',)),
 ('away', ('3',)),
 ('bourgeois', ('3',)),
 ('bourgeoisie', ('3',)),
 ('but', ('3',)),
 ('camps', ('3',)),
 ('cats', ('1',)),
 ('class', ('3',)),
 ('classes', ('3',)),
 ('communism', ('2',)),
 ('conditions', ('3',)),
 ('czar', ('2',)),
 ('directly', ('3',)),
 ('distinctive', ('3',))]

In [10]:
final = counts.map(lambda x: ((x[0],len(x[1])), x[1]))

output = final.take(20)
for i in range(20):
    print(output[i])

(('a', 2), ('2', '3'))
(('all', 1), ('2',))
(('alliance', 1), ('2',))
(('and', 3), ('1', '2', '3'))
(('antagonisms', 1), ('3',))
(('are', 1), ('1',))
(('as', 1), ('3',))
(('away', 1), ('3',))
(('bourgeois', 1), ('3',))
(('bourgeoisie', 1), ('3',))
(('but', 1), ('3',))
(('camps', 1), ('3',))
(('cats', 1), ('1',))
(('class', 1), ('3',))
(('classes', 1), ('3',))
(('communism', 1), ('2',))
(('conditions', 1), ('3',))
(('czar', 1), ('2',))
(('directly', 1), ('3',))
(('distinctive', 1), ('3',))
